## Traficom Summary Merge Preparation
This notebook prepares the price data and cleaned Traficom summary tables for merge by loading the cleaned summary files, checking merge-key quality, repairing merge keys in `price_df`, and validating that the datasets are ready for a later merge.

## Step 1 — Load datasets

In [ ]:
from pathlib import Path
from IPython.display import display
import pandas as pd

base_dir = Path("/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM")

price_df = pd.read_csv(base_dir / "datasets/cleaned/cleaned_and_merged_pricedataset.csv")
brand_summary_df = pd.read_csv(base_dir / "datasets/traficom_outputs/brand_summary_clean.csv")
model_summary_df = pd.read_csv(base_dir / "datasets/traficom_outputs/model_summary_clean.csv")

## Step 2 — Basic inspection

In [ ]:
print(f"price_df shape: {price_df.shape}")
print(f"brand_summary_df shape: {brand_summary_df.shape}")
print(f"model_summary_df shape: {model_summary_df.shape}")

print("\nRelevant price_df columns:")
print([col for col in ["product_id", "part_name", "brand", "model", "category", "subcategory"] if col in price_df.columns])

print("\nRelevant brand_summary_df columns:")
print([col for col in ["brand", "brand_total_registered"] if col in brand_summary_df.columns])

print("\nRelevant model_summary_df columns:")
print([col for col in ["brand", "model_family_clean", "model_total_registered"] if col in model_summary_df.columns])

## Step 3 — Standardize merge-key text

In [ ]:
def standardize_key(series):
    series = series.astype("string").str.strip().str.lower()
    return series.replace({"": pd.NA, "nan": pd.NA})

for df in [price_df, brand_summary_df, model_summary_df]:
    df.columns = df.columns.str.strip()

price_df["brand"] = standardize_key(price_df["brand"])
price_df["model"] = standardize_key(price_df["model"])
price_df["category"] = standardize_key(price_df["category"])
price_df["subcategory"] = standardize_key(price_df["subcategory"])

brand_summary_df["brand"] = standardize_key(brand_summary_df["brand"])

model_summary_df["brand"] = standardize_key(model_summary_df["brand"])
model_summary_df["model_family_clean"] = standardize_key(model_summary_df["model_family_clean"])

## Step 4 — Check merge-key completeness and uniqueness

In [ ]:
missing_key_summary_df = pd.DataFrame(
    {
        "dataset": ["price_df", "model_summary_df", "brand_summary_df"],
        "missing_brand": [
            int(price_df["brand"].isna().sum()),
            int(model_summary_df["brand"].isna().sum()),
            int(brand_summary_df["brand"].isna().sum()),
        ],
        "missing_model": [
            int(price_df["model"].isna().sum()),
            int(model_summary_df["model_family_clean"].isna().sum()),
            pd.NA,
        ],
    }
)
display(missing_key_summary_df)

print("Duplicate brand keys in brand_summary_df:", int(brand_summary_df[["brand"]].dropna().duplicated().sum()))
print(
    "Duplicate brand-model keys in model_summary_df:",
    int(model_summary_df[["brand", "model_family_clean"]].dropna().duplicated().sum()),
)

## Step 5 — Diagnose `price_df` merge-key consistency

In [ ]:
known_manufacturer_set = set(brand_summary_df["brand"].dropna().unique())
known_model_family_set = set(model_summary_df["model_family_clean"].dropna().unique())

price_df["category_key"] = price_df["category"]
price_df["subcategory_key"] = price_df["subcategory"]

part_taxonomy_terms = set(price_df["category_key"].dropna()) | set(price_df["subcategory_key"].dropna())
part_keyword_pattern = r"engine|brake|fuel|airbag|suspension|gear ?box|axle|electric|databox|sensor|transmitter|vehicle exterior|drive axle|middle axle"

price_df["model_looks_like_part_taxonomy"] = (
    price_df["model"].eq(price_df["category_key"])
    | price_df["model"].eq(price_df["subcategory_key"])
    | price_df["model"].isin(part_taxonomy_terms)
    | price_df["model"].astype("string").str.contains(part_keyword_pattern, case=False, na=False, regex=True)
)
price_df["brand_is_known_model_family"] = price_df["brand"].isin(known_model_family_set)

diagnostic_summary_df = pd.DataFrame(
    {
        "diagnostic": [
            "brand is known manufacturer",
            "brand is known model family",
            "model is known model family",
            "model looks like part taxonomy",
        ],
        "row_count": [
            int(price_df["brand"].isin(known_manufacturer_set).sum()),
            int(price_df["brand_is_known_model_family"].sum()),
            int(price_df["model"].isin(known_model_family_set).sum()),
            int(price_df["model_looks_like_part_taxonomy"].sum()),
        ],
    }
)
diagnostic_summary_df["share_of_rows"] = diagnostic_summary_df["row_count"] / len(price_df)
display(diagnostic_summary_df)

suspicious_pairs_df = (
    price_df.loc[
        price_df["brand_is_known_model_family"] | price_df["model_looks_like_part_taxonomy"],
        ["brand", "model"],
    ]
    .value_counts()
    .rename("row_count")
    .reset_index()
)
display(suspicious_pairs_df.head(10))

## Step 6 — Repair merge keys safely

In [ ]:
manufacturer_alias_map = {"vw": "volkswagen"}

model_family_brand_counts = (
    model_summary_df[["brand", "model_family_clean"]]
    .dropna()
    .drop_duplicates()
    .groupby("model_family_clean")["brand"]
    .nunique()
)
unique_model_brand_map = (
    model_summary_df[["brand", "model_family_clean"]]
    .dropna()
    .drop_duplicates()
    .groupby("model_family_clean")["brand"]
    .first()
    .loc[model_family_brand_counts.eq(1)]
)
ambiguous_model_family_set = set(model_family_brand_counts[model_family_brand_counts.gt(1)].index)

traficom_pair_index = pd.MultiIndex.from_frame(
    model_summary_df[["brand", "model_family_clean"]].dropna().drop_duplicates()
)

price_df["brand_canonical_candidate"] = price_df["brand"].replace(manufacturer_alias_map)
price_df["brand_merge_key"] = pd.NA
price_df["model_merge_key"] = pd.NA
price_df["repair_status"] = "unknown_brand_model"

original_valid_mask = pd.MultiIndex.from_arrays(
    [price_df["brand_canonical_candidate"], price_df["model"]]
).isin(traficom_pair_index)

price_df.loc[original_valid_mask, "brand_merge_key"] = price_df.loc[original_valid_mask, "brand_canonical_candidate"]
price_df.loc[original_valid_mask, "model_merge_key"] = price_df.loc[original_valid_mask, "model"]
price_df.loc[original_valid_mask, "repair_status"] = "original_valid"

repair_from_model_family_mask = (
    ~original_valid_mask
    & price_df["brand"].isin(unique_model_brand_map.index)
    & price_df["model_looks_like_part_taxonomy"]
)
price_df.loc[repair_from_model_family_mask, "brand_merge_key"] = price_df.loc[
    repair_from_model_family_mask, "brand"
].map(unique_model_brand_map)
price_df.loc[repair_from_model_family_mask, "model_merge_key"] = price_df.loc[
    repair_from_model_family_mask, "brand"
]
price_df.loc[repair_from_model_family_mask, "repair_status"] = "repaired_from_known_model_family"

ambiguous_mask = (
    price_df["repair_status"].eq("unknown_brand_model")
    & price_df["brand"].isin(ambiguous_model_family_set)
    & price_df["model_looks_like_part_taxonomy"]
)
price_df.loc[ambiguous_mask, "repair_status"] = "ambiguous_unrepaired"

invalid_taxonomy_mask = (
    price_df["repair_status"].eq("unknown_brand_model")
    & price_df["model_looks_like_part_taxonomy"]
)
price_df.loc[invalid_taxonomy_mask, "repair_status"] = "invalid_part_taxonomy_in_model"

## Step 7 — Validate repaired merge keys

In [ ]:
original_pair_index = pd.MultiIndex.from_frame(price_df[["brand", "model"]])
repaired_pair_index = pd.MultiIndex.from_frame(price_df[["brand_merge_key", "model_merge_key"]])

matched_row_share_before = original_pair_index.isin(traficom_pair_index).mean()
matched_row_share_after = repaired_pair_index.isin(traficom_pair_index).mean()

comparison_summary_df = pd.DataFrame(
    {
        "metric": [
            "matched row share",
            "non-missing repaired key rows",
        ],
        "before": [round(float(matched_row_share_before), 4), int(price_df[["brand", "model"]].dropna().shape[0])],
        "after": [round(float(matched_row_share_after), 4), int(price_df[["brand_merge_key", "model_merge_key"]].dropna().shape[0])],
    }
)
display(comparison_summary_df)

repair_status_summary_df = (
    price_df["repair_status"]
    .value_counts(dropna=False)
    .rename_axis("repair_status")
    .reset_index(name="row_count")
)
display(repair_status_summary_df)

top_repaired_pairs_df = (
    price_df[["brand_merge_key", "model_merge_key"]]
    .dropna()
    .value_counts()
    .rename("row_count")
    .reset_index()
)
display(top_repaired_pairs_df.head(10))

## Step 8 — Final merge-readiness check

In [ ]:
brand_key_duplicates = int(brand_summary_df[["brand"]].dropna().duplicated().sum())
model_key_duplicates = int(
    model_summary_df[["brand", "model_family_clean"]].dropna().duplicated().sum()
)

print("Duplicate brand keys in brand_summary_df:", brand_key_duplicates)
print("Duplicate brand-model keys in model_summary_df:", model_key_duplicates)
print("Recommended validate setting:", "many_to_one" if model_key_duplicates == 0 else "not safe for many_to_one")

test_merge_df = price_df[[col for col in ["product_id", "brand_merge_key", "model_merge_key", "repair_status"] if col in price_df.columns]].copy()
test_merge_df = test_merge_df.dropna(subset=["brand_merge_key", "model_merge_key"])

test_merge_result_df = test_merge_df.merge(
    model_summary_df[["brand", "model_family_clean"]].drop_duplicates(),
    how="left",
    left_on=["brand_merge_key", "model_merge_key"],
    right_on=["brand", "model_family_clean"],
    indicator=True,
    validate="many_to_one",
)

print("\nTest merge indicator counts")
print(test_merge_result_df["_merge"].value_counts(dropna=False).to_string())

## Step 9 — Short conclusion
- The cleaned Traficom summary CSVs now load normally and are the files used in this notebook.
- The Traficom summary tables are unique on their intended merge keys.
- The original `price_df` merge keys were not merge-ready because model-family and part-taxonomy text were mixed into the raw fields.
- Conservative repaired merge keys were created in `brand_merge_key` and `model_merge_key` while preserving the original columns.
- The notebook is now ready for the actual merge step in a separate notebook or next section, but it does not perform the final permanent merge yet.